In [136]:
from sklearn.preprocessing import StandardScaler
from dataclasses import dataclass, field
from sklearn.linear_model import Ridge
from datetime import datetime
from fredapi import Fred
import yfinance as yf
import pandas as pd
import numpy as np
import os

In [141]:
class DataStore:
  def __init__(self, debug: bool = False):
    self.debug = debug

  def get_data(self, tickers: list, start: str, end: str, fred_api_key: str,
               interval: str = "1d", benchmark: str = "^GSPC",
               cyclical: str = "XLY", defensive: str = "XLP"):
    tmp = []
    for t in tickers:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)
      else:
        print(f"Warning: Skipping {t} | type: {type(t)}")
    tickers_clean = sorted(set(tmp))

    df_path = f"prices_{start}_{end}.parquet"
    if os.path.exists(df_path):
      df = pd.read_parquet(df_path)
    else:
      df = yf.download(tickers_clean, start=start, end=end, interval=interval)["Close"]
      df.to_parquet(df_path)

    returns = df.ffill().pct_change(fill_method=None)

    factors = self._get_factors(start, end, fred_api_key, benchmark, cyclical, defensive)
    return returns, factors

  def _fetch_equity_factors(self, start, end, benchmark="^GSPC",
                            cyclical="XLY", defensive="XLP"):
    r = yf.download([benchmark, "^IRX", cyclical, defensive],
                    start=start, end=end)["Close"]
    rets = r[[benchmark, cyclical, defensive]].pct_change(fill_method=None)
    rf_daily = (r["^IRX"].ffill() / 100) / 252

    factors = pd.DataFrame({
      "Benchmark": rets[benchmark] - rf_daily,
      "Spread": rets[cyclical] - rets[defensive],
    })
    return factors

  def _get_breakeven_inf(self, fred) -> pd.Series:
    return fred.get_series("DGS5") - fred.get_series("DFII5")

  def _fetch_fred(self, fred_api_key) -> pd.DataFrame:
    fred = Fred(api_key=fred_api_key)
    cols = {
      "NFCI":         fred.get_series("NFCI"),
      "USD":          fred.get_series("DTWEXBGS"),
      "TIPS10Y":      fred.get_series("DFII10"),
      "HighYieldOAS": fred.get_series("BAMLH0A0HYM2"),
      "BreakevenInf": self._get_breakeven_inf(fred),
    }
    out = pd.concat(cols, axis=1).sort_index()
    out.index = pd.to_datetime(out.index)
    return out

  def _get_factors(self, start, end, fred_api_key, benchmark="^GSPC",
                   cyclical="XLY", defensive="XLP") -> pd.DataFrame:
    path = f"factors_{start}_{end}.parquet"
    if os.path.exists(path):
      return pd.read_parquet(path)

    fred_df = self._fetch_fred(fred_api_key)
    eq_df = self._fetch_equity_factors(start, end, benchmark, cyclical, defensive)

    combined = pd.concat([fred_df, eq_df], axis=1, join="outer").sort_index()
    combined = combined.loc[start:end]
    combined.to_parquet(path)

    if self.debug:
      print("First valid date per factor:")
      print(combined.apply(pd.Series.first_valid_index))
    return combined

In [152]:
@dataclass
class BootstrapParams:
  n_bootstraps: int = 1000
  block_length: int = 10
  horizon: int = None
  circular: bool = False
  seed: int = None

  def __post_init__(self):
    if self.n_bootstraps < 1:
      raise ValueError(f"Invalid n_bootstraps: {self.n_bootstraps}")

    if self.block_length < 1:
      raise ValueError(f"Invalid block_length: {self.block_length}")

    if self.horizon is not None and self.horizon < 1:
      raise ValueError(f"Invalid horizon: {self.horizon}")

In [153]:
class BlockBootstrap:
  def __init__(self, params: BootstrapParams = None):
    self.p = params or BootstrapParams()
    self.rng = np.random.default_rng(self.p.seed)

  def _validate(self, returns: pd.DataFrame) -> int:
    if returns.isna().any().any():
      raise ValueError("returns contains NaNs; clean/align them first (e.g. dropna()).")

    T = len(returns)
    if self.p.block_length > T:
      raise ValueError(f"block_length ({self.p.block_length}) exceeds sample length ({T})")

    return self.p.horizon if self.p.horizon is not None else T

  def _sample_indices(self, T: int, horizon: int) -> np.ndarray:
    L = self.p.block_length
    n_blocks = int(np.ceil(horizon / L))

    n_starts = T if self.p.circular else T - L + 1
    starts = self.rng.integers(0, n_starts, size=(self.p.n_bootstraps, n_blocks))

    idx = (starts[:, :, None] + np.arange(L)).reshape(self.p.n_bootstraps, -1)
    idx = idx[:, :horizon]

    return idx % T if self.p.circular else idx

  def resample(self, returns: pd.DataFrame, as_dataframe: bool = False):
    horizon = self._validate(returns)
    values = returns.to_numpy()

    idx = self._sample_indices(len(values), horizon)
    samples = values[idx]

    if not as_dataframe:
      return samples

    index = pd.MultiIndex.from_product(
      [range(self.p.n_bootstraps), range(horizon)], names=["bootstrap", "step"]
    )
    return pd.DataFrame(
      samples.reshape(-1, samples.shape[2]), index=index, columns=returns.columns
    )

In [150]:
FACTOR_KINDS = {
  "Benchmark":    "compound",
  "Spread":       "compound",
  "USD":          "pct",
  "NFCI":         "diff",
  "BreakevenInf": "diff",
  "HighYieldOAS": "diff",
  "TIPS10Y":      "diff",
}
@dataclass
class FilterParams:
  min_length: tuple = (5, "y")
  debug: bool = True
  factor_kinds: dict = field(default_factory=lambda: dict(FACTOR_KINDS))
  week_anchor: str = "W-FRI"
  min_days_per_week: int = 3
  min_weeks: int = 26
  window_size: int = 104
  ridge_alpha: float = 1.0

  def __post_init__(self):
    if not self.min_length[1] in ["d", "m", "y"]:
      raise ValueError(f"Invalid time period: {self.min_length[1]}")

    if self.min_length[0] <= 0:
      raise ValueError(f"Invalid min length: {self.min_length[0]}")

    if not 1 <= self.min_days_per_week <= 7:
      raise ValueError(f"Invalid min_days_per_week: {self.min_days_per_week}")

    if self.min_weeks <= 0:
      raise ValueError(f"Invalid min_weeks: {self.min_weeks}")

In [ ]:
class Filter(BlockBootstrap):
  def __init__(self, filter_params: FilterParams, bootstrap_params: BootstrapParams):
    self.p = filter_params
    super().__init__(bootstrap_params)

  @staticmethod
  def _naive(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df.sort_index()

  def _compound_weekly(self, rets: pd.DataFrame) -> pd.DataFrame:
    g = np.log1p(rets).resample(self.p.week_anchor)
    weekly = np.expm1(g.sum(min_count=1))
    return weekly.where(rets.resample(self.p.week_anchor).count() >= self.p.min_days_per_week)

  def _to_weekly_returns(self, returns: pd.DataFrame) -> pd.DataFrame:
    return self._compound_weekly(self._naive(returns))

  def _to_weekly_factors(self, factors: pd.DataFrame) -> pd.DataFrame:
    factors = self._naive(factors)
    kinds = self.p.factor_kinds

    unknown = set(factors.columns) - set(kinds)
    if unknown:
      raise ValueError(f"No aggregation rule for factors: {sorted(unknown)}")

    comp = [c for c in factors.columns if kinds[c] == "compound"]
    lvl = [c for c in factors.columns if kinds[c] != "compound"]

    parts = []
    if comp:
      parts.append(self._compound_weekly(factors[comp]))
    if lvl:
      wk = factors[lvl].resample(self.p.week_anchor).last().ffill(limit=1)
      for c in lvl:
        wk[c] = wk[c].diff() if kinds[c] == "diff" else wk[c].pct_change(fill_method=None)
      parts.append(wk)

    return pd.concat(parts, axis=1)[factors.columns]

  def _time_horizon_filter(self, returns: pd.DataFrame) -> pd.DataFrame:
    end = pd.to_datetime(returns.index.max())
    min_l, tp = self.p.min_length

    min_start = end - pd.DateOffset(
      years=min_l if tp == "y" else 0,
      months=min_l if tp == "m" else 0,
      days=min_l if tp == "d" else 0,
    )
    if self.p.debug:
      print(f"Min start: {min_start}")

    drop = [t for t in returns.columns
            if pd.to_datetime(returns[t].dropna().index.min()) > min_start]
    return returns.drop(columns=drop).dropna()

  def _get_bootstraps(self, returns: pd.DataFrame, factors: pd.DataFrame):
    common = returns.dropna().index.intersection(factors.dropna().index)
    Y_df = returns.loc[common]
    X_df = pd.DataFrame(StandardScaler().fit_transform(factors.loc[common]),
                        index=common, columns=factors.columns)

    assets, fnames = Y_df.columns, X_df.columns
    N, K = len(assets), len(fnames)

    draws = self.resample(pd.concat([Y_df, X_df], axis=1))
    Yb, Xb = draws[:, :, :N], draws[:, :, N:]

    coefs = np.empty((self.p.n_bootstraps, N, K))
    for b in range(self.p.n_bootstraps):
      coefs[b] = self._fit_ridge(X=Xb[b], y=Yb[b]).coef_

    lower = np.percentile(coefs, 2.5, axis=0)
    upper = np.percentile(coefs, 97.5, axis=0)

    p_pos = (coefs > 0).mean(axis=0)
    stability = np.maximum(p_pos, 1 - p_pos)

    factor_index = pd.Index(fnames, name="Factor")

    percentile_intervals = pd.DataFrame(
      np.stack([lower, upper], axis=2).transpose(1, 0, 2).reshape(K, N * 2),
      index=factor_index,
      columns=pd.MultiIndex.from_product(
        [assets, ["Lower CI (2.5%)", "Upper CI (97.5%)"]],
        names=["Asset", "Bound"],
      ),
    )

    sign_stability = pd.DataFrame(
      stability.T,
      index=factor_index,
      columns=pd.Index(assets, name="Asset"),
    )

    return percentile_intervals, sign_stability

  def _fit_ridge(self, X: pd.DataFrame, y: pd.Series) -> pd.Series:
    model = Ridge(alpha=1.0)
    model.fit(X, y)
    return model

  def _rolling_factor_params(self, returns: pd.DataFrame, factors: pd.DataFrame):
    if returns.isna().any().any() or factors.isna().any().any():
      raise ValueError("returns/factors contain NaNs; align and clean them first.")

    idx = returns.index.intersection(factors.index)
    returns, factors = returns.loc[idx], factors.loc[idx]

    factors_scaled = pd.DataFrame(StandardScaler().fit_transform(factors),
                                  index=factors.index, columns=factors.columns)

    w = self.p.window_size
    T, N, K = len(idx), returns.shape[1], factors.shape[1]
    Y = returns.to_numpy()

    betas = np.full((T, N, K), np.nan)
    r2_in = np.full((T, N), np.nan)
    r2_oos = np.full((T, N), np.nan)
    oos_resid = np.full((T, N), np.nan)
    rmse_oos = np.full((T, N), np.nan)

    sse = np.zeros(N)
    sst = np.zeros(N)

    for i in range(w, T):
      X_win = factors_scaled.iloc[i - w : i]
      y_win = Y[i - w : i]

      model = self._fit_ridge(X=X_win, y=returns.iloc[i - w : i])
      betas[i] = model.coef_

      resid = y_win - model.predict(X_win)
      y_mean = y_win.mean(axis=0)
      with np.errstate(divide="ignore", invalid="ignore"):
        r2_in[i] = 1 - (resid ** 2).sum(axis=0) / ((y_win - y_mean) ** 2).sum(axis=0)

      y_hat = model.predict(factors_scaled.iloc[[i]])[0]
      oos_resid[i] = Y[i] - y_hat
      sse += oos_resid[i] ** 2
      sst += (Y[i] - y_mean) ** 2
      n_oos = i - w + 1
      rmse_oos[i] = np.sqrt(sse / n_oos)

      with np.errstate(divide="ignore", invalid="ignore"):
        r2_oos[i] = 1 - sse / sst

    resid_vol = (pd.DataFrame(oos_resid)
                  .expanding(min_periods=2)
                  .std(ddof=0)
                  .to_numpy())

    betas_df = pd.DataFrame(
      betas.reshape(T, N * K),
      index=idx,
      columns=pd.MultiIndex.from_product([returns.columns, factors.columns],
                                        names=["Asset", "Factor"]),
    )

    metrics = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol"]
    stats_df = pd.DataFrame(
      np.stack(
          [r2_in, r2_oos, rmse_oos, resid_vol], axis=2
      ).reshape(T, N * len(metrics)),
      index=idx,
      columns=pd.MultiIndex.from_product([returns.columns, metrics],
                                        names=["Asset", "Metric"]),
    )

    return betas_df, stats_df

  def _factor_exposure_filter(
    self,
    returns: pd.DataFrame,
    factors: pd.DataFrame
  ) -> tuple[pd.DataFrame, pd.DataFrame]:
    weekly_returns = self._to_weekly_returns(returns)
    weekly_factors = self._to_weekly_factors(factors).dropna()

    common = weekly_returns.index.intersection(weekly_factors.index)
    X = weekly_factors.loc[common]
    Y = weekly_returns.loc[common]

    percentile_intervals, sign_stability = self._get_bootstraps(Y, X)
    betas, stats_df = self._rolling_factor_params(Y, X)

    factor_fields = pd.concat(
      [percentile_intervals.stack("Asset"),
      sign_stability.stack().rename("SignStability")],
      axis=1,
    ) 
    factor_part = pd.concat(
      {f: factor_fields.xs(f, level="Factor") for f in X.columns}, axis=1
    )  
    factor_part.columns.names = ["Factor", "Field"]

    model_cols = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol"]
    model_part = pd.concat(
      {"Model": stats_df.iloc[-1].unstack("Metric")[model_cols]}, axis=1
    )
    model_part.columns.names = ["Factor", "Field"]

    summary = pd.concat([factor_part, model_part], axis=1)

    return betas, summary



    

